# Phase 1 — The Dataset
## Brain Tumour MRI Classification

Everything that decides what the model is trained on and measured against
happens here, and nothing downstream is allowed to revisit it.

The previous version of this project discovered, after a full write-up, that its
model was reading file provenance rather than anatomy. Every gate in this phase
exists to catch that class of problem before a single epoch is trained.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import collections

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src import audit, config, data, manifest, splits, viz
from src.config import CHENG_DIR, CLASSES, IMG_SIZE, SEED, TEST_FRAC, VAL_FRAC
from src.splits import _hash

print(f"source     {CHENG_DIR}")
print(f"classes    {list(CLASSES)}")
print(f"split      test {TEST_FRAC:.0%} of all, then val {VAL_FRAC:.0%} of the rest")

source     C:\Games\Codes\Python\Projects\Brain_Tumour_Detection\Final_Project\data\raw\cheng
classes    ['glioma', 'meningioma', 'pituitary']
split      test 20% of all, then val 15% of the rest


In [ ]:
# 1. WHAT THIS DATASET IS, AND WHY IT REPLACED THE LAST ONE
"""
The figshare collection published by Cheng et al. (2017): 3,064 T1-weighted
contrast-enhanced slices from 233 patients, one acquisition protocol across two
hospitals, every image 512x512. Each file is MATLAB v7.3 -- which is HDF5, so
h5py rather than scipy.io -- and carries three things the previous dataset did
not have.

"""
D = splits.build_dataset(CHENG_DIR)
images, labels, groups = D["images"], D["labels"], D["groups"]
masks, names, pids = D["masks"], D["names"], D["pids"]

counts = collections.Counter(labels.tolist())
print(f"{'class':<14}{'images':>8}{'share':>9}")
print("-" * 31)
for i, c in enumerate(CLASSES):
    print(f"{c:<14}{counts[i]:>8}{counts[i]/len(labels):>9.1%}")
print("-" * 31)
print(f"{'total':<14}{len(labels):>8}")

per = collections.Counter(groups.tolist())
print(f"\npatients            {int(D['n_patients'])}")
print(f"slices per patient  mean {np.mean(list(per.values())):.1f}, "
      f"median {int(np.median(list(per.values())))}, max {max(per.values())}")
print(f"imbalance           {counts.most_common()[0][1] / counts.most_common()[-1][1]:.2f}:1")
print(f"cache               {images.shape}, uint8")

class           images    share
-------------------------------
glioma            1426    46.5%
meningioma         708    23.1%
pituitary          930    30.4%
-------------------------------
total             3064

patients            233
slices per patient  mean 13.2, median 13, max 38
imbalance           2.01:1
cache               (3064, 224, 224), uint8


In [ ]:
# 2. NOTHING IS CACHED, DELIBERATELY

S = splits.build_splits(images, labels, groups)
train_idx, val_idx, test_idx = S["train_idx"], S["val_idx"], S["test_idx"]
DATASET_HASH = _hash(images, labels, groups)
print(f"dataset hash  {DATASET_HASH}")
print(f"split hash    {S['split_hash']}")
print("both are pure functions of the .mat files and the seed, so they are stable")
print("across runs without anything being stored between them")

dataset hash  bc41cbf1f869ebb1
split hash    02b0799a48a71b69
both are pure functions of the .mat files and the seed, so they are stable
across runs without anything being stored between them


In [ ]:
# 3. THE AUDIT GATE, BEFORE ANY TRAINING
D_eq = splits.build_dataset(CHENG_DIR, equalise=True)

print("  probe                             accuracy   chance     lift")
audit.texture_probe(images, labels, list(CLASSES), label="3-class, raw")
audit.texture_probe(D_eq["images"], labels, list(CLASSES), label="3-class, equalised")
del D_eq

  probe                             accuracy   chance     lift
  3-class, raw                        0.6521   0.3333  +0.3187   top: laplacian var
  3-class, equalised                  0.6472   0.3333  +0.3138   top: gradient


In [5]:
# 4. WHAT THE PROBE MEANS, MEASURED RATHER THAN ASSUMED
"""
The probe sits well above chance, and the reason matters more than the number.

Two possibilities. Either acquisition is leaking -- some artefact of how the
images were made correlates with the label -- or class genuinely correlates with
anatomy, and statistics that track slice level inherit some of that.

They are distinguishable. Tumour type really does predict location: pituitary
tumours sit in the sella at the skull base, meningiomas are extra-axial,
gliomas are intra-axial and infiltrative. So the test is whether the ground-truth
mask's geometry -- centroid, area, extent, and nothing else, no pixels at all --
predicts class at least as well as the texture probe does. If it does, anatomy
is a sufficient explanation and no acquisition artefact needs to be invoked.

Note what this does NOT establish: that the model uses anatomy rather than
texture. That is a question about the model, and Phase 5 answers it causally by
occluding the lesion.
"""
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score

def probe(X, name):
    a = cross_val_score(RandomForestClassifier(300, random_state=SEED, n_jobs=-1),
                        X, labels, cv=GroupKFold(5), groups=groups).mean()
    print(f"  {name:<44}{a:>8.4f}")
    return a

anat = np.array([[xs.mean()/m.shape[1], ys.mean()/m.shape[0], m.sum()/m.size,
                  np.ptp(xs)/m.shape[1], np.ptp(ys)/m.shape[0]]
                 for m in masks for ys, xs in [np.nonzero(m)]])
brain = np.array([[b.mean(), np.ptp(xs)/im.shape[1], np.ptp(ys)/im.shape[0]]
                  for im in images
                  for b in [im > 10] for ys, xs in [np.nonzero(b)]])

print(f"  {'probe (patient-wise folds)':<44}{'accuracy':>8}")
print("  " + "-" * 52)
print(f"  {'chance':<44}{1/len(CLASSES):>8.4f}")
print(f"  {'majority class':<44}{np.bincount(labels).max()/len(labels):>8.4f}")
A_ANAT = probe(anat, "mask centroid + area + extent (pure anatomy)")
probe(brain, "brain extent only (slice-level proxy)")
A_TEX = probe(audit.texture_features(images), "texture, 5 pixel statistics")

print(f"""
Anatomy alone reaches {A_ANAT:.4f}, ABOVE the texture probe's {A_TEX:.4f}. Class really is
largely a function of where the tumour is, so the texture lift needs no
acquisition artefact to explain it. This is the baseline Phase 5 reports the
model against: {A_TEX:.4f} is what five global statistics achieve, and the margin
over it is the part attributable to learned structure.""")

  probe (patient-wise folds)                  accuracy
  ----------------------------------------------------
  chance                                        0.3333
  majority class                                0.4654
  mask centroid + area + extent (pure anatomy)  0.7063
  brain extent only (slice-level proxy)         0.5111
  texture, 5 pixel statistics                   0.6691

Anatomy alone reaches 0.7063, ABOVE the texture probe's 0.6691. Class really is
largely a function of where the tumour is, so the texture lift needs no
acquisition artefact to explain it. This is the baseline Phase 5 reports the
model against: 0.6691 is what five global statistics achieve, and the margin
over it is the part attributable to learned structure.


In [ ]:
# 5. PREPROCESSING, AND WHAT THE MASK HAS TO FOLLOW
"""
Each slice is min-max scaled to 8 bits, cropped to its brain extent, and resized
to a 224px cache. The working resolution is 128px; caching larger leaves room to
ablate resolution without decoding again.
"""
from src.sources import load_cheng
sample_files = sorted(Path(CHENG_DIR).glob("*.mat"))[:4]
raw_img, raw_lab, _, raw_msk, _ = load_cheng(sample_files)

fig, axes = viz.styled_fig(2, 4, figsize=(11, 6))
for j in range(len(sample_files)):
    axes[0, j].imshow(raw_img[j], cmap='gray')
    axes[0, j].set_title(f"{raw_lab[j]}", fontsize=9)
    axes[1, j].imshow(raw_img[j], cmap='gray')
    axes[1, j].imshow(np.ma.masked_where(~raw_msk[j], raw_msk[j]),
                      cmap='autumn', alpha=0.55)
    axes[1, j].set_title("mask, same geometry", fontsize=9)
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle("Cached image and its annotation, after identical processing",
             fontsize=12, fontweight='bold')
plt.tight_layout(); viz.save(fig, "preprocessing.png")

area = masks.reshape(len(masks), -1).sum(1)
inside = np.array([im[m].mean() for im, m in zip(images, masks)])
outside = np.array([im[~m].mean() for im, m in zip(images, masks)])
print(f"images with a non-empty mask   {(area > 0).sum()} / {len(masks)}")
print(f"mask area, share of frame      mean {(area/masks[0].size).mean():.4f}, "
      f"min {(area/masks[0].size).min():.4f}, max {(area/masks[0].size).max():.4f}")
print(f"mean intensity inside vs out   {inside.mean():.1f} vs {outside.mean():.1f} "
      f"({(inside-outside).mean():+.1f})")
assert (area > 0).all(), "a tumour image has an empty mask"
print("\nevery image has a lesion mask, and lesions are brighter than their")
print("surroundings, which is what a contrast-enhanced sequence should show")

  saved -> outputs/preprocessing.png
images with a non-empty mask   3064 / 3064
mask area, share of frame      mean 0.0242, min 0.0011, max 0.1232
mean intensity inside vs out   87.7 vs 50.1 (+37.5)

every image has a lesion mask, and lesions are brighter than their
surroundings, which is what a contrast-enhanced sequence should show


In [ ]:
# 6. DUPLICATES: LOOKED FOR, AND NOT FOUND

print(f"duplicate pairs found          {int((~S['keep']).sum())}")
print(f"patient groups merged          {int(S['n_group_merges'])}")
print(f"images kept                    {int(S['keep'].sum())} of {len(labels)}")
ex = splits.write_exclusions(names, S)
print(f"\nwritten to outputs/excluded.json: {ex['counts']}")
print("""
Zero. This collection is clean, and that is a property of the data rather than
of the cleaning -- the check is identical to the one that removed 293 images
from the previous dataset. Reporting the number as zero is worth more than not
running the check, because it is the same measurement either way.""")

duplicate pairs found          0
patient groups merged          0
images kept                    3064 of 3064

written to outputs/excluded.json: {'dropped_as_duplicate': 0, 'kept': 3064}

Zero. This collection is clean, and that is a property of the data rather than
of the cleaning -- the check is identical to the one that removed 293 images
from the previous dataset. Reporting the number as zero is worth more than not
running the check, because it is the same measurement either way.


In [ ]:
# 7. THE SPLIT, BY PATIENT
"""
Cheng ships no train/test division, so one is drawn here -- twice, both times
grouped by patient. Test is carved off first from the whole pool and never
touched again; validation is then carved from what remains. So the test fraction
is of everything and the validation fraction is of the remainder, which is
stated in config rather than left to be inferred from the sizes.

"""
print(f"{'split':<8}{'images':>8}{'patients':>10}   " +
      "  ".join(f"{c}" for c in CLASSES))
print("-" * 62)
for name, ix in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
    c = collections.Counter(labels[ix].tolist())
    print(f"{name:<8}{len(ix):>8}{len(set(S['merged_groups'][ix].tolist())):>10}   " +
          "  ".join(f"{c[i]:>{len(cl)}}" for i, cl in enumerate(CLASSES)))
print("-" * 62)
print(f"{'total':<8}{len(train_idx)+len(val_idx)+len(test_idx):>8}"
      f"{int(D['n_patients']):>10}")

viz.plot_class_balance([collections.Counter(labels[train_idx].tolist())[i]
                        for i in range(len(CLASSES))], list(CLASSES))

split     images  patients   glioma  meningioma  pituitary
--------------------------------------------------------------
train       2099       159      977         485        637
val          352        27      163          81        108
test         613        47      286         142        185
--------------------------------------------------------------
total       3064       233
  saved -> outputs/class_balance.png


WindowsPath('C:/Games/Codes/Python/Projects/Brain_Tumour_Detection/Final_Project/outputs/class_balance.png')

In [ ]:
# 8. THE GATES
"""
The leakage gate re-derives its check from the images rather than reusing the
mask that produced the exclusion: a check that trusts the thing it is checking
verifies nothing.
"""
pat = splits.assert_patient_disjoint(S["merged_groups"], train_idx, val_idx, test_idx)
print(f"PASS  no patient in two splits        train {pat['train']}, "
      f"val {pat['val']}, test {pat['test']}")

lk = splits.assert_no_leakage(images, train_idx, test_idx)
print(f"PASS  no duplicate test<->train       {lk['checked']} checked, "
      f"max cosine {lk['max_cosine']:.4f}")

nbr = splits.neighbour_report(images, labels, train_idx, test_idx)
print(f"\n{'nearest training image':<32}{'same class':>12}{'other class':>13}")
print("-" * 57)
for t in (0.99, 0.98, 0.95, 0.90):
    print(f"  cosine >= {t:.2f}{'':<18}{nbr['bands'][t]:>12.3%}"
          f"{nbr['null_bands'][t]:>13.3%}")
print("""
The right-hand column is the different-patient null: the nearest training image
of a DIFFERENT class, which is almost certainly a different patient. The two
columns sitting close together is the evidence that patient grouping worked.""")

PASS  no patient in two splits        train 159, val 27, test 47
PASS  no duplicate test<->train       613 checked, max cosine 0.9588

nearest training image            same class  other class
---------------------------------------------------------
  cosine >= 0.99                        0.000%       0.000%
  cosine >= 0.98                        0.000%       0.000%
  cosine >= 0.95                        0.163%       0.000%
  cosine >= 0.90                        0.653%       0.163%

The right-hand column is the different-patient null: the nearest training image
of a DIFFERENT class, which is almost certainly a different patient. The two
columns sitting close together is the evidence that patient grouping worked.


In [ ]:
# 9. NORMALISATION, FROM THE TRAINING SPLIT ONLY
MEAN, STD = data.compute_stats(images, train_idx)
print(f"mean {MEAN:.6f}   std {STD:.6f}   from {len(train_idx)} training images")

all_mean, all_std = data.compute_stats(images, np.arange(len(images)))
print(f"\nfor comparison, over the whole dataset: mean {all_mean:.6f}, std {all_std:.6f}")
print(f"difference: {abs(MEAN-all_mean):.6f} / {abs(STD-all_std):.6f} -- small, which is")
print("why using the wrong one would never have been noticed from the numbers")

mean 0.198996   std 0.158494   from 2099 training images

for comparison, over the whole dataset: mean 0.199470, std 0.159034
difference: 0.000474 / 0.000540 -- small, which is
why using the wrong one would never have been noticed from the numbers


In [ ]:
# 10. AUGMENTATION
tf = data.make_transforms(MEAN, STD, augment=True, img_size=IMG_SIZE)
i = int(train_idx[0])
fig, axes = viz.styled_fig(2, 4, figsize=(11, 6))
axes[0, 0].imshow(images[i], cmap='gray'); axes[0, 0].set_title("cached", fontsize=9)
for j, ax in enumerate(axes.ravel()[1:], 1):
    ax.imshow(viz.denorm(tf(Image.fromarray(images[i], mode="L")), MEAN, STD),
              cmap='gray')
    ax.set_title(f"augmented {j}", fontsize=9)
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle("Training augmentation \u2014 affine and intensity jitter, no flip",
             fontsize=12, fontweight='bold')
plt.tight_layout(); viz.save(fig, "augmentation.png")
print("affine (rotate/translate/scale) plus brightness-contrast jitter, no flip")

  saved -> outputs/augmentation.png
affine (rotate/translate/scale) plus brightness-contrast jitter, no flip


In [12]:
# 11. SUMMARY, AND THE RUN MANIFEST
"""
The manifest is written here because this is where the dataset and the split are
fixed. Its hash is stamped on every figure and embedded in every checkpoint, so
a figure and a model can be checked against each other instead of assumed to
match. They once did not, silently, and nothing on disk said so.
"""
MANIFEST = manifest.write(DATASET_HASH, S["split_hash"], extra={
    "equalise": False,
    "n_images": int(len(labels)),
    "n_patients": int(D["n_patients"]),
    "split_sizes": {k: int(len(v)) for k, v in
                    (("train", train_idx), ("val", val_idx), ("test", test_idx))},
    "patients_per_split": {k: int(len(set(S["merged_groups"][v].tolist())))
                           for k, v in (("train", train_idx), ("val", val_idx),
                                        ("test", test_idx))},
    "texture_probe_patientwise": round(float(A_TEX), 4),
    "anatomy_oracle_patientwise": round(float(A_ANAT), 4),
})

checks = [
    ("every image has a lesion mask",     bool((masks.reshape(len(masks), -1).sum(1) > 0).all())),
    ("no duplicates anywhere",            int((~S["keep"]).sum()) == 0),
    ("no patient in two splits",          True),
    ("no duplicate test<->train",         True),
    ("normalisation from train only",     len(train_idx) < len(images)),
    ("all three classes in every split",  all(
        len(set(labels[ix].tolist())) == len(CLASSES)
        for ix in (train_idx, val_idx, test_idx))),
    ("nothing cached to disk",            not (config.ROOT / "data" / "cache").exists()),
]
print("=" * 64)
print("PHASE 1 VERIFICATION")
print("=" * 64)
for label, ok in checks:
    print(f"  {'OK  ' if ok else 'FAIL'}  {label}")
failed = [l for l, ok in checks if not ok]
assert not failed, "failed checks: " + "; ".join(failed)

print(f"""
  dataset        Cheng et al. 2017, {len(labels)} slices, {int(D['n_patients'])} patients
  classes        {', '.join(CLASSES)}
  split          train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)}
  patients       {pat['train']} / {pat['val']} / {pat['test']}, disjoint
  duplicates     {int((~S['keep']).sum())} found, {int(S['n_group_merges'])} patient groups merged
  normalisation  mean {MEAN:.4f}, std {STD:.4f}, training split only
  texture probe  {A_TEX:.4f} patient-wise, against {1/len(CLASSES):.4f} chance
  anatomy oracle {A_ANAT:.4f} -- above the probe, so anatomy explains it
  run manifest   {MANIFEST}

  Phase 2 defines the model. The test set is not opened until Phase 5.""")

PHASE 1 VERIFICATION
  OK    every image has a lesion mask
  OK    no duplicates anywhere
  OK    no patient in two splits
  OK    no duplicate test<->train
  OK    normalisation from train only
  OK    all three classes in every split
  OK    nothing cached to disk

  dataset        Cheng et al. 2017, 3064 slices, 233 patients
  classes        glioma, meningioma, pituitary
  split          train 2099 / val 352 / test 613
  patients       159 / 27 / 47, disjoint
  duplicates     0 found, 0 patient groups merged
  normalisation  mean 0.1990, std 0.1585, training split only
  texture probe  0.6691 patient-wise, against 0.3333 chance
  anatomy oracle 0.7063 -- above the probe, so anatomy explains it
  run manifest   b9dcd147c12e

  Phase 2 defines the model. The test set is not opened until Phase 5.
